# Textbook OCR Pipeline — Google Colab

교재 사진·PDF를 업로드하면 영문 본문, 2단 레이아웃, 선이 있는 표와 그림 영역을 분석합니다. 표는 CSV로, 그림은 PNG와 내부 라벨 TXT로 저장하며 전체 결과를 ZIP으로 내려받습니다.

In [ ]:
# 1. Tesseract와 최신 프로젝트 설치
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-kor
!pip -q install --upgrade 'git+https://github.com/SKKUPhysicsKing/textbook-ocr-pipeline.git@main'
!tesseract --list-langs

In [ ]:
# 2. 설정
import shutil
import zipfile
from pathlib import Path
from google.colab import files
from textbook_ocr.pipeline import PipelineConfig, run_pipeline
from textbook_ocr.preprocess import PreprocessConfig

LANGUAGE = 'eng'       # 한영 혼합 문서는 'kor+eng'
LAYOUT = 'auto'        # auto, single, columns
VISUALS = 'detect'     # detect, off
UPSCALE_WIDTH = 0      # 저해상도 비교 실험 시 1600
SAVE_PROCESSED = True

WORK_DIR = Path('/content/textbook_ocr')
INPUT_DIR = WORK_DIR / 'input'
OUTPUT_DIR = WORK_DIR / 'output'
shutil.rmtree(WORK_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)
print('설정 완료')

In [ ]:
# 3. 이미지, PDF 또는 ZIP 업로드
uploaded = files.upload()
for original_name, data in uploaded.items():
    destination = INPUT_DIR / Path(original_name).name
    destination.write_bytes(data)
    if destination.suffix.lower() == '.zip':
        extract_dir = INPUT_DIR / destination.stem
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(destination) as archive:
            for member in archive.infolist():
                target = (extract_dir / member.filename).resolve()
                if extract_dir.resolve() not in target.parents and target != extract_dir.resolve():
                    raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
            archive.extractall(extract_dir)
print(f'{len(uploaded)}개 파일 업로드 완료')

In [ ]:
# 4. OCR 실행
config = PipelineConfig(
    language=LANGUAGE,
    layout=LAYOUT,
    visuals=VISUALS,
    save_processed=SAVE_PROCESSED,
    preprocess=PreprocessConfig(mode='raw', upscale_width=UPSCALE_WIDTH),
)
manifest = run_pipeline(INPUT_DIR, OUTPUT_DIR, config=config)
table_count = sum(v['kind'] == 'table' for page in manifest['pages'] for v in page['visuals'])
figure_count = sum(v['kind'] == 'figure' for page in manifest['pages'] for v in page['visuals'])
print(f"완료: {manifest['page_count']}쪽, 표 {table_count}개, 그림 {figure_count}개")

In [ ]:
# 5. Markdown 결과 미리보기
combined = (OUTPUT_DIR / 'combined.md').read_text(encoding='utf-8')
print(combined[:6000])

In [ ]:
# 6. 결과 ZIP 다운로드
archive_path = Path(shutil.make_archive('/content/textbook_ocr_results', 'zip', OUTPUT_DIR))
print(f'다운로드: {archive_path.name}')
files.download(str(archive_path))